# Clinical-trial negative efficacy classifier

Google Colab notebook for a **binary text classifier** over the `ST1` sheet:

- `1` = trial stopped due to **negative efficacy**
- `0` = all other stop reasons

Implemented methods:
1. TF–IDF + Logistic Regression
2. DistilBERT
3. BiomedBERT / PubMedBERT-lineage model

Confidence outputs:
- calibrated probabilities
- tuned decision threshold
- split-conformal prediction sets
- optional MC-dropout uncertainty for the best transformer

References:
- https://www.nature.com/articles/s41588-024-01854-z
- https://huggingface.co/distilbert/distilbert-base-uncased
- https://huggingface.co/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext

In [ ]:
!pip -q install -U transformers accelerate datasets scikit-learn joblib openpyxl

In [ ]:
import os
import gc
import json
import math
import random
import shutil
import warnings
import re
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.special import softmax
from sklearn.calibration import calibration_curve
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

In [ ]:
SHEET_NAME = "ST1"
HEADER_ROW = 2
TEXT_COL = "Reasons for termination"
LABEL_COLS = ["Reason1", "Reason2", "Reason3"]
STATUS_COL = "Status"
CATEGORY_COL = "Category"

POSITIVE_REASON_LABELS = {
    "Insufficient Efficacy",
    "Futility",
    "Unmet endpoint",
}

RELAXED_EXTRA_POSITIVE_LABELS = {
    # Example if you later want it:
    # "Based on data in this study"
}

DROP_CONFLICTING_DUPLICATE_TEXTS = True
DROP_EMPTY_TEXTS = True

TEST_SIZE = 0.20
VAL_SIZE = 0.10
CAL_SIZE = 0.10

MAX_LEN = 96
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

MODELS_TO_RUN = {
    "logreg": True,
    "distilbert": True,
    "biomedbert": True,
}

HF_MODELS = {
    "distilbert": "distilbert/distilbert-base-uncased",
    "biomedbert": "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
}

EXPORT_DIR = Path("/content/negative_efficacy_export") if IN_COLAB else Path("./negative_efficacy_export")
LOCAL_WORKBOOK_PATH = None
LOGREG_C_GRID = [2.0, 4.0]


In [ ]:
def resolve_input_path(local_path=None):
    if local_path:
        p = Path(local_path)
        if p.exists():
            return p

    candidates = [
        Path("/content/41588_2024_1854_MOESM3_ESM.xlsx"),
        Path("./41588_2024_1854_MOESM3_ESM.xlsx"),
        Path("/mnt/data/41588_2024_1854_MOESM3_ESM.xlsx"),
    ]
    for p in candidates:
        if p.exists():
            return p

    if IN_COLAB:
        print("Upload the workbook now...")
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No workbook uploaded.")
        return Path(next(iter(uploaded.keys())))

    raise FileNotFoundError("Workbook not found. Set LOCAL_WORKBOOK_PATH or upload the file.")

WORKBOOK_PATH = resolve_input_path(LOCAL_WORKBOOK_PATH)
print("Using workbook:", WORKBOOK_PATH)

raw_df = pd.read_excel(WORKBOOK_PATH, sheet_name=SHEET_NAME, header=HEADER_ROW)
display(raw_df.head())
print(raw_df.shape)
print(raw_df.columns.tolist())

In [ ]:
def normalize_text(text):
    text = "" if pd.isna(text) else str(text)
    text = text.replace("&amp;", "&")
    text = re.sub(r"\s+", " ", text.strip().lower())
    return text

def derive_binary_label(row, positive_labels):
    labels = {row.get(col) for col in LABEL_COLS if pd.notna(row.get(col))}
    return int(any(label in positive_labels for label in labels))

LOW_INFO_PATTERNS = [
    r"^see detailed description\.?$",
    r"^see termination reason in detailed description\.?$",
    r"^please see detailed description(?: below)? for termination reason\.?$",
]

def is_low_information_text(text_norm):
    return any(re.match(pat, text_norm) for pat in LOW_INFO_PATTERNS)

def build_binary_dataset(df, positive_labels, drop_conflicting_duplicate_texts=True):
    required = set(LABEL_COLS + [TEXT_COL, STATUS_COL, CATEGORY_COL])
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing expected columns: {sorted(missing)}")

    work = df.copy()

    if DROP_EMPTY_TEXTS:
        work = work.loc[work[TEXT_COL].notna()].copy()
        work = work.loc[work[TEXT_COL].astype(str).str.strip().ne("")].copy()

    work["text"] = work[TEXT_COL].astype(str)
    work["text_norm"] = work["text"].map(normalize_text)
    work["label"] = work.apply(lambda row: derive_binary_label(row, positive_labels), axis=1).astype(int)
    work["low_information_text"] = work["text_norm"].map(is_low_information_text)

    work["reason_labels"] = work[LABEL_COLS].apply(
        lambda row: sorted({v for v in row.values.tolist() if pd.notna(v)}),
        axis=1,
    )

    label_nunique_by_text = work.groupby("text_norm")["label"].nunique()
    conflict_texts = set(label_nunique_by_text[label_nunique_by_text > 1].index)
    work["text_conflict"] = work["text_norm"].isin(conflict_texts)

    if drop_conflicting_duplicate_texts:
        work = work.loc[~work["text_conflict"]].copy()

    work = work.reset_index(drop=True)
    return work, conflict_texts

binary_df, conflict_texts = build_binary_dataset(
    raw_df,
    POSITIVE_REASON_LABELS | RELAXED_EXTRA_POSITIVE_LABELS,
    drop_conflicting_duplicate_texts=DROP_CONFLICTING_DUPLICATE_TEXTS,
)

print("Rows after preprocessing:", len(binary_df))
print("Positive rows:", int(binary_df["label"].sum()))
print("Positive rate:", round(float(binary_df["label"].mean()), 4))
print("Conflicting normalized texts removed:", len(conflict_texts))
print("Low-information rows flagged:", int(binary_df["low_information_text"].sum()))
display(binary_df[[TEXT_COL, "label", "reason_labels", "low_information_text"]].head(10))

In [ ]:
print("Category distribution")
display(binary_df[CATEGORY_COL].value_counts(dropna=False).to_frame("count"))

print("Status distribution")
display(binary_df[STATUS_COL].value_counts(dropna=False).to_frame("count"))

print("Positive examples")
display(
    binary_df.loc[binary_df["label"] == 1, [TEXT_COL, "reason_labels"]]
    .sample(min(12, int(binary_df["label"].sum())), random_state=SEED)
)

print("Conflicting texts removed (sample)")
for x in list(sorted(conflict_texts))[:10]:
    print("-", x)

In [ ]:
def make_group_splits(df, group_col="text_norm", label_col="label", test_size=0.2, val_size=0.1, cal_size=0.1, seed=42):
    if test_size + val_size + cal_size >= 1.0:
        raise ValueError("test_size + val_size + cal_size must be < 1.0")

    groups = df[[group_col, label_col]].drop_duplicates().reset_index(drop=True)

    train_val_cal_groups, test_groups = train_test_split(
        groups,
        test_size=test_size,
        random_state=seed,
        stratify=groups[label_col],
    )

    remaining_after_test = 1.0 - test_size
    val_rel = val_size / remaining_after_test

    train_cal_groups, val_groups = train_test_split(
        train_val_cal_groups,
        test_size=val_rel,
        random_state=seed,
        stratify=train_val_cal_groups[label_col],
    )

    remaining_after_test_val = remaining_after_test - val_size
    cal_rel = cal_size / remaining_after_test_val

    train_groups, cal_groups = train_test_split(
        train_cal_groups,
        test_size=cal_rel,
        random_state=seed,
        stratify=train_cal_groups[label_col],
    )

    split_map = {}
    for split_name, split_part in {
        "train": train_groups,
        "val": val_groups,
        "cal": cal_groups,
        "test": test_groups,
    }.items():
        for key in split_part[group_col]:
            split_map[key] = split_name

    out = df.copy()
    out["split"] = out[group_col].map(split_map)
    assert out["split"].isna().sum() == 0
    return out

split_df = make_group_splits(
    binary_df,
    group_col="text_norm",
    label_col="label",
    test_size=TEST_SIZE,
    val_size=VAL_SIZE,
    cal_size=CAL_SIZE,
    seed=SEED,
)

for split_name in ["train", "val", "cal", "test"]:
    sub = split_df.loc[split_df["split"] == split_name]
    print(
        split_name,
        {
            "rows": len(sub),
            "positive_rows": int(sub["label"].sum()),
            "positive_rate": round(float(sub["label"].mean()), 4),
            "unique_groups": sub["text_norm"].nunique(),
        },
    )

train_df = split_df.loc[split_df["split"] == "train"].reset_index(drop=True)
val_df = split_df.loc[split_df["split"] == "val"].reset_index(drop=True)
cal_df = split_df.loc[split_df["split"] == "cal"].reset_index(drop=True)
test_df = split_df.loc[split_df["split"] == "test"].reset_index(drop=True)

In [ ]:
def expected_calibration_error(y_true, y_prob, n_bins=10):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(y_prob, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    ece = 0.0
    for i in range(n_bins):
        mask = bin_ids == i
        if not np.any(mask):
            continue
        ece += np.abs(y_true[mask].mean() - y_prob[mask].mean()) * mask.mean()
    return float(ece)

def best_f1_threshold(y_true, y_prob):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)

    thresholds = np.unique(np.concatenate([np.linspace(0.01, 0.99, 199), y_prob]))
    best_t = 0.5
    best_score = -1.0
    for t in thresholds:
        preds = (y_prob >= t).astype(int)
        score = f1_score(y_true, preds, zero_division=0)
        if score > best_score:
            best_score = score
            best_t = float(t)
    return best_t, float(best_score)

def compute_binary_metrics(y_true, y_prob, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "average_precision": average_precision_score(y_true, y_prob),
        "brier": brier_score_loss(y_true, y_prob),
        "ece_10": expected_calibration_error(y_true, y_prob, n_bins=10),
        "threshold": threshold,
    }
    try:
        out["roc_auc"] = roc_auc_score(y_true, y_prob)
    except Exception:
        out["roc_auc"] = np.nan
    return out

def plot_model_diagnostics(y_true, y_prob, threshold=0.5, title_prefix=""):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob).astype(float)
    y_pred = (y_prob >= threshold).astype(int)

    fig, axes = plt.subplots(1, 4, figsize=(24, 5))

    try:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        axes[0].plot(fpr, tpr)
        axes[0].plot([0, 1], [0, 1], linestyle="--")
        axes[0].set_title(f"{title_prefix} ROC")
        axes[0].set_xlabel("FPR")
        axes[0].set_ylabel("TPR")
    except Exception:
        axes[0].text(0.5, 0.5, "ROC unavailable", ha="center", va="center")

    prec, rec, _ = precision_recall_curve(y_true, y_prob)
    axes[1].plot(rec, prec)
    axes[1].set_title(f"{title_prefix} PR")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")

    frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy="uniform")
    axes[2].plot(mean_pred, frac_pos, marker="o")
    axes[2].plot([0, 1], [0, 1], linestyle="--")
    axes[2].set_title(f"{title_prefix} Reliability")

    cm = confusion_matrix(y_true, y_pred)
    axes[3].imshow(cm)
    axes[3].set_title(f"{title_prefix} Confusion @ {threshold:.3f}")
    axes[3].set_xticks([0, 1], labels=["Other", "Neg efficacy"])
    axes[3].set_yticks([0, 1], labels=["Other", "Neg efficacy"])
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            axes[3].text(j, i, str(cm[i, j]), ha="center", va="center")

    plt.tight_layout()
    plt.show()

class SigmoidScoreCalibrator:
    def __init__(self):
        self.model = LogisticRegression(solver="lbfgs")

    def fit(self, scores, y):
        self.model.fit(np.asarray(scores).reshape(-1, 1), np.asarray(y).astype(int))
        return self

    def predict_proba(self, scores):
        return self.model.predict_proba(np.asarray(scores).reshape(-1, 1))[:, 1]

def fit_split_conformal(prob_pos, y_true, alpha=0.10):
    prob_pos = np.asarray(prob_pos).astype(float)
    y_true = np.asarray(y_true).astype(int)
    true_prob = np.where(y_true == 1, prob_pos, 1.0 - prob_pos)
    nonconformity = 1.0 - true_prob
    n = len(nonconformity)
    q = min(math.ceil((n + 1) * (1 - alpha)) / n, 1.0)
    try:
        qhat = np.quantile(nonconformity, q, method="higher")
    except TypeError:
        qhat = np.quantile(nonconformity, q, interpolation="higher")
    return float(qhat)

def conformal_prediction_sets(prob_pos, qhat):
    prob_pos = np.asarray(prob_pos).astype(float)
    out = []
    for p1 in prob_pos:
        p0 = 1.0 - p1
        pred_set = []
        if 1.0 - p0 <= qhat:
            pred_set.append(0)
        if 1.0 - p1 <= qhat:
            pred_set.append(1)
        out.append(pred_set)
    return out

In [ ]:
RESULTS = {}
ARTIFACTS = {}

In [ ]:
def train_logistic_baseline(train_df, val_df, cal_df, test_df):
    vectorizer = FeatureUnion([
        ("word", TfidfVectorizer(
            lowercase=True,
            strip_accents="unicode",
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True,
        )),
        ("char", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            sublinear_tf=True,
        )),
    ])

    X_train = vectorizer.fit_transform(train_df["text"])
    X_val = vectorizer.transform(val_df["text"])
    X_cal = vectorizer.transform(cal_df["text"])
    X_test = vectorizer.transform(test_df["text"])

    y_train = train_df["label"].to_numpy()
    y_val = val_df["label"].to_numpy()
    y_cal = cal_df["label"].to_numpy()
    y_test = test_df["label"].to_numpy()

    best = None
    for c in LOGREG_C_GRID:
        clf = LogisticRegression(C=c, class_weight="balanced", max_iter=4000, solver="liblinear")
        clf.fit(X_train, y_train)
        val_prob = clf.predict_proba(X_val)[:, 1]
        ap = average_precision_score(y_val, val_prob)
        if best is None or ap > best["val_ap"]:
            best = {"C": c, "clf": clf, "val_ap": ap}

    clf = best["clf"]
    cal_scores = clf.decision_function(X_cal)
    calibrator = SigmoidScoreCalibrator().fit(cal_scores, y_cal)

    val_prob = calibrator.predict_proba(clf.decision_function(X_val))
    test_prob = calibrator.predict_proba(clf.decision_function(X_test))
    threshold, _ = best_f1_threshold(y_val, val_prob)
    metrics = compute_binary_metrics(y_test, test_prob, threshold=threshold)

    cal_prob = calibrator.predict_proba(cal_scores)
    qhat = fit_split_conformal(cal_prob, y_cal, alpha=0.10)

    artifacts = {
        "type": "logreg",
        "vectorizer": vectorizer,
        "model": clf,
        "calibrator": calibrator,
        "threshold": threshold,
        "qhat": qhat,
        "positive_reason_labels": sorted(POSITIVE_REASON_LABELS | RELAXED_EXTRA_POSITIVE_LABELS),
    }
    aux = {
        "test_prob": test_prob,
        "y_test": y_test,
        "best_C": best["C"],
    }
    return metrics, artifacts, aux

if MODELS_TO_RUN["logreg"]:
    metrics, artifacts, aux = train_logistic_baseline(train_df, val_df, cal_df, test_df)
    RESULTS["logreg"] = metrics | {"model_name": "TFIDF+LogReg", "best_C": aux["best_C"]}
    ARTIFACTS["logreg"] = artifacts
    display(pd.DataFrame([RESULTS["logreg"]]).T)
    plot_model_diagnostics(aux["y_test"], aux["test_prob"], threshold=artifacts["threshold"], title_prefix="TFIDF+LogReg")

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=96):
        self.texts = list(texts)
        self.labels = None if labels is None else list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return item

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        weight = None if self.class_weights is None else self.class_weights.to(logits.device)
        loss = nn.CrossEntropyLoss(weight=weight)(logits, labels)
        return (loss, outputs) if return_outputs else loss

def hf_compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax(logits, axis=1)[:, 1]
    preds = (probs >= 0.5).astype(int)
    out = {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, zero_division=0),
        "recall": recall_score(labels, preds, zero_division=0),
        "f1": f1_score(labels, preds, zero_division=0),
        "average_precision": average_precision_score(labels, probs),
    }
    try:
        out["roc_auc"] = roc_auc_score(labels, probs)
    except Exception:
        out["roc_auc"] = np.nan
    return out

def predict_logits(model, tokenizer, texts, batch_size=32, max_length=96):
    ds = TextDataset(texts, labels=None, tokenizer=tokenizer, max_length=max_length)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False)
    model.eval()
    all_logits = []
    with torch.no_grad():
        for batch in dl:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)
            all_logits.append(outputs.logits.detach().cpu().numpy())
    return np.vstack(all_logits)

def fit_temperature(logits, y_true, max_iter=50):
    logits_t = torch.tensor(logits, dtype=torch.float32)
    labels_t = torch.tensor(np.asarray(y_true).astype(int), dtype=torch.long)
    temperature = nn.Parameter(torch.ones(1))
    optimizer = torch.optim.LBFGS([temperature], lr=0.01, max_iter=max_iter)
    nll = nn.CrossEntropyLoss()

    def closure():
        optimizer.zero_grad()
        loss = nll(logits_t / temperature.clamp(min=1e-3), labels_t)
        loss.backward()
        return loss

    optimizer.step(closure)
    return float(temperature.detach().clamp(min=1e-3).cpu().item())

def apply_temperature(logits, temperature):
    return softmax(logits / float(temperature), axis=1)[:, 1]

In [ ]:
def train_transformer_model(model_key, model_name, train_df, val_df, cal_df, test_df):
    seed_everything(SEED)
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
    model.to(DEVICE)

    train_ds = TextDataset(train_df["text"], train_df["label"], tokenizer, max_length=MAX_LEN)
    val_ds = TextDataset(val_df["text"], val_df["label"], tokenizer, max_length=MAX_LEN)

    class_counts = train_df["label"].value_counts().sort_index()
    total = int(class_counts.sum())
    class_weights = torch.tensor(
        [total / max(int(class_counts.get(0, 1)), 1), total / max(int(class_counts.get(1, 1)), 1)],
        dtype=torch.float32,
    )

    output_dir = Path("./trainer_outputs") / model_key
    output_dir.mkdir(parents=True, exist_ok=True)

    args = TrainingArguments(
        output_dir=str(output_dir),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_average_precision",
        greater_is_better=True,
        report_to="none",
        seed=SEED,
        fp16=torch.cuda.is_available(),
        dataloader_pin_memory=torch.cuda.is_available(),
    )

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        tokenizer=tokenizer,
        class_weights=class_weights,
        compute_metrics=hf_compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )
    trainer.train()

    val_logits = predict_logits(model, tokenizer, val_df["text"].tolist(), batch_size=EVAL_BATCH_SIZE, max_length=MAX_LEN)
    cal_logits = predict_logits(model, tokenizer, cal_df["text"].tolist(), batch_size=EVAL_BATCH_SIZE, max_length=MAX_LEN)
    test_logits = predict_logits(model, tokenizer, test_df["text"].tolist(), batch_size=EVAL_BATCH_SIZE, max_length=MAX_LEN)

    temperature = fit_temperature(cal_logits, cal_df["label"].to_numpy())
    val_prob = apply_temperature(val_logits, temperature)
    cal_prob = apply_temperature(cal_logits, temperature)
    test_prob = apply_temperature(test_logits, temperature)

    threshold, _ = best_f1_threshold(val_df["label"].to_numpy(), val_prob)
    metrics = compute_binary_metrics(test_df["label"].to_numpy(), test_prob, threshold=threshold)
    qhat = fit_split_conformal(cal_prob, cal_df["label"].to_numpy(), alpha=0.10)

    artifacts = {
        "type": "transformer",
        "model_key": model_key,
        "model_name": model_name,
        "tokenizer": tokenizer,
        "model": model,
        "temperature": temperature,
        "threshold": threshold,
        "qhat": qhat,
        "positive_reason_labels": sorted(POSITIVE_REASON_LABELS | RELAXED_EXTRA_POSITIVE_LABELS),
    }
    aux = {
        "test_prob": test_prob,
        "y_test": test_df["label"].to_numpy(),
    }
    return metrics, artifacts, aux

In [ ]:
if MODELS_TO_RUN["distilbert"]:
    metrics, artifacts, aux = train_transformer_model(
        "distilbert",
        HF_MODELS["distilbert"],
        train_df,
        val_df,
        cal_df,
        test_df,
    )
    RESULTS["distilbert"] = metrics | {"model_name": HF_MODELS["distilbert"]}
    ARTIFACTS["distilbert"] = artifacts
    display(pd.DataFrame([RESULTS["distilbert"]]).T)
    plot_model_diagnostics(aux["y_test"], aux["test_prob"], threshold=artifacts["threshold"], title_prefix="DistilBERT")

In [ ]:
if MODELS_TO_RUN["biomedbert"]:
    metrics, artifacts, aux = train_transformer_model(
        "biomedbert",
        HF_MODELS["biomedbert"],
        train_df,
        val_df,
        cal_df,
        test_df,
    )
    RESULTS["biomedbert"] = metrics | {"model_name": HF_MODELS["biomedbert"]}
    ARTIFACTS["biomedbert"] = artifacts
    display(pd.DataFrame([RESULTS["biomedbert"]]).T)
    plot_model_diagnostics(aux["y_test"], aux["test_prob"], threshold=artifacts["threshold"], title_prefix="BiomedBERT")

In [ ]:
results_df = pd.DataFrame(RESULTS).T.sort_values(by=["average_precision", "f1", "roc_auc"], ascending=False)
display(results_df)
BEST_MODEL_KEY = results_df.index[0]
print("Selected best model:", BEST_MODEL_KEY)

In [ ]:
def _score_logreg(artifact, texts):
    X = artifact["vectorizer"].transform(list(texts))
    raw_scores = artifact["model"].decision_function(X)
    prob_pos = artifact["calibrator"].predict_proba(raw_scores)
    return prob_pos, raw_scores

def _score_transformer(artifact, texts):
    logits = predict_logits(
        artifact["model"],
        artifact["tokenizer"],
        list(texts),
        batch_size=EVAL_BATCH_SIZE,
        max_length=MAX_LEN,
    )
    prob_pos = apply_temperature(logits, artifact["temperature"])
    margin = np.abs(prob_pos - 0.5) * 2.0
    return prob_pos, margin

def predict_texts(texts, model_key=None):
    if model_key is None:
        model_key = BEST_MODEL_KEY

    artifact = ARTIFACTS[model_key]
    texts = list(texts)

    if artifact["type"] == "logreg":
        prob_pos, secondary = _score_logreg(artifact, texts)
        confidence_name = "raw_log_odds"
    else:
        prob_pos, secondary = _score_transformer(artifact, texts)
        confidence_name = "margin_from_0.5"

    pred_sets = conformal_prediction_sets(prob_pos, artifact["qhat"])
    point_preds = (prob_pos >= artifact["threshold"]).astype(int)

    rows = []
    for text, p1, pred, pred_set, sec in zip(texts, prob_pos, point_preds, pred_sets, secondary):
        if pred_set == [1]:
            decision = "predict_negative_efficacy"
        elif pred_set == [0]:
            decision = "predict_other_reason"
        else:
            decision = "review"

        rows.append({
            "text": text,
            "prob_negative_efficacy": float(p1),
            "point_prediction": int(pred),
            "prediction_set": str(pred_set),
            "decision": decision,
            confidence_name: float(sec),
        })
    return pd.DataFrame(rows)

demo_texts = [
    "Study was terminated after interim analysis showed futility and lack of efficacy.",
    "Trial stopped because of slow recruitment and lack of funding.",
    "Please see detailed description for termination reason.",
]
display(predict_texts(demo_texts))

In [ ]:
def enable_dropout(model):
    for module in model.modules():
        if module.__class__.__name__.startswith("Dropout"):
            module.train()

def mc_dropout_predict_proba(model, tokenizer, texts, n_passes=20, batch_size=32, max_length=96):
    ds = TextDataset(texts, labels=None, tokenizer=tokenizer, max_length=max_length)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=False)

    probs_passes = []
    model.eval()
    enable_dropout(model)

    with torch.no_grad():
        for _ in range(n_passes):
            probs_this_pass = []
            for batch in dl:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                logits = model(**batch).logits
                probs_this_pass.append(torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy())
            probs_passes.append(np.concatenate(probs_this_pass))

    probs_passes = np.stack(probs_passes, axis=0)
    return probs_passes.mean(axis=0), probs_passes.std(axis=0)

if BEST_MODEL_KEY in {"distilbert", "biomedbert"}:
    mean_prob, std_prob = mc_dropout_predict_proba(
        ARTIFACTS[BEST_MODEL_KEY]["model"],
        ARTIFACTS[BEST_MODEL_KEY]["tokenizer"],
        demo_texts,
        n_passes=10,
        batch_size=8,
        max_length=MAX_LEN,
    )
    display(pd.DataFrame({
        "text": demo_texts,
        "mc_dropout_mean_prob": mean_prob,
        "mc_dropout_std": std_prob,
    }))
else:
    print("Best model is not a transformer; skipping MC dropout.")

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

def export_artifacts(results_df, export_dir=EXPORT_DIR):
    export_dir.mkdir(parents=True, exist_ok=True)

    summary = {
        "sheet_name": SHEET_NAME,
        "text_col": TEXT_COL,
        "label_cols": LABEL_COLS,
        "positive_reason_labels": sorted(POSITIVE_REASON_LABELS | RELAXED_EXTRA_POSITIVE_LABELS),
        "best_model_key": BEST_MODEL_KEY,
        "results": results_df.reset_index(names="model_key").to_dict(orient="records"),
    }
    with open(export_dir / "run_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    for model_key, artifact in ARTIFACTS.items():
        if artifact["type"] == "logreg":
            joblib.dump({
                "vectorizer": artifact["vectorizer"],
                "model": artifact["model"],
                "calibrator": artifact["calibrator"],
                "threshold": artifact["threshold"],
                "qhat": artifact["qhat"],
                "positive_reason_labels": artifact["positive_reason_labels"],
            }, export_dir / f"{model_key}_bundle.joblib")
        else:
            model_dir = export_dir / model_key
            model_dir.mkdir(exist_ok=True, parents=True)
            artifact["model"].save_pretrained(model_dir)
            artifact["tokenizer"].save_pretrained(model_dir)
            with open(model_dir / "posthoc_config.json", "w") as f:
                json.dump({
                    "model_name": artifact["model_name"],
                    "temperature": artifact["temperature"],
                    "threshold": artifact["threshold"],
                    "qhat": artifact["qhat"],
                    "positive_reason_labels": artifact["positive_reason_labels"],
                }, f, indent=2)

    zip_path = shutil.make_archive(str(export_dir), "zip", export_dir)
    return Path(zip_path)

zip_path = export_artifacts(results_df, EXPORT_DIR)
print("Exported:", zip_path)

if IN_COLAB:
    files.download(str(zip_path))

## Notes

- Default label policy is **conservative**:
  - positive = any of `Insufficient Efficacy`, `Futility`, `Unmet endpoint`
  - negative = everything else
- The notebook drops normalized texts that map to both classes, because they are not identifiable from the single sentence alone.
- Confidence is exposed through:
  - calibrated probability
  - tuned threshold
  - conformal prediction set
  - optional MC-dropout spread for the best transformer

## References

- https://www.nature.com/articles/s41588-024-01854-z
- https://huggingface.co/distilbert/distilbert-base-uncased
- https://huggingface.co/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext